In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import seaborn as sns
import pandas as pd
_here = Path.cwd().resolve()
_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_root / "src"))
from regression_modelling.models.cv import (
    full_coverage_cities, load_pooled_table, loco_folds,
)

In [2]:
# Load 5 cities with comprehensive crime incident coverage
cities = full_coverage_cities()            # ['houston','chicago','atlanta','kansas_city','detroit']
# daytime_pop_floor=100 (default) drops small-denominator rate ARTIFACTS — a few crimes
# over ~no daytime people — the diagnosed target-stabilization treatment. Genuine hotspots
# (hundreds of crimes over a solid daytime population) sit far above the floor and are kept.
pooled = load_pooled_table(daytime_pop_floor=100)

# inspect the LOCO split
for city, train, holdout in loco_folds(pooled):
    print(city, "→ train", train.shape, "holdout", holdout.shape)


  houston            1629 BGs
  chicago            2164 BGs
  atlanta             426 BGs
  kansas_city         473 BGs
  detroit             622 BGs
  pooled             5314 BGs across 5 cities
  dropped 22 zero/NaN-pop BGs -> 5292 remain (filtered geoid set for fit + Moran's I + bias join)
  dropped 8 BGs below daytime_pop 100 (small-denominator rate artifacts) -> 5284 remain
atlanta → train (4860, 82) holdout (424, 82)
chicago → train (3125, 82) holdout (2159, 82)
detroit → train (4679, 82) holdout (605, 82)
houston → train (3657, 82) holdout (1627, 82)
kansas_city → train (4815, 82) holdout (469, 82)


### Component 2 — leakage-safe fit / predict core

One design matrix, several target forms. The predictor scaler is fit on the **train cities
only** and applied to the holdout, so no holdout information touches the fit.

- `rate_daytime_within_city` — **headline (c)**: per-city z-scored *daytime* rate
  (count / (population + LODES jobs) per 1k), i.e. relative within-city risk on the
  stabilized denominator
- `rate_daytime` — **reported (a)**: raw daytime rate (absolute level)
- `logcount` — comparator: log(count+1)

Demonstrated below on a **single fold** (hold out one city). The full LOCO loop is the next
component.

In [3]:
from scipy.stats import spearmanr
from regression_modelling.models.cv import (
    fit_fold, predict_fold, make_target, TARGET_MODES,
)

# one fold: hold out a chosen city, train on the other four
HOLDOUT = "chicago"
folds = {c: (tr, ho) for c, tr, ho in loco_folds(pooled)}
train, holdout = folds[HOLDOUT]
print(f"holdout={HOLDOUT}  train={train.shape}  holdout={holdout.shape}")

holdout=chicago  train=(3125, 82)  holdout=(2159, 82)


In [4]:
holdout.info()

<class 'pandas.DataFrame'>
Index: 2159 entries, 1629 to 3792
Data columns (total 82 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   city                              2159 non-null   str    
 1   geoid                             2159 non-null   str    
 2   det_pct                           2159 non-null   float64
 3   moved1yr_pct                      2159 non-null   float64
 4   own_pct                           2159 non-null   float64
 5   lap_pct                           2159 non-null   float64
 6   Division                          2159 non-null   float64
 7   city_centers_dist                 2159 non-null   float64
 8   pop_est_5mile                     2159 non-null   float64
 9   pop_ch_1mile                      2159 non-null   float64
 10  vacant_pct                        2159 non-null   float64
 11  clip_liens_pct                    2159 non-null   float64
 12  clip_foreclosure_pc

In [5]:
# fit each target mode on the train fold, score the held-out city
rows = []
scored_by_mode = {}
for mode in TARGET_MODES:
    fit = fit_fold(train, mode=mode)
    scored = predict_fold(fit, holdout)
    scored_by_mode[mode] = scored
    rows.append({
        "mode": mode,
        "n_train": fit["n_train"],
        "adj_r2_in_sample": round(fit["result"].rsquared_adj, 3),
        "holdout_n": len(scored),
        # ranking sanity: predicted risk vs ACTUAL holdout rate (headline metric is rank-based)
        "spearman_pred_vs_daytime_rate": round(spearmanr(scored["y_pred"], scored["cl_total_rate_daytime"]).statistic, 3),
    })
pd.DataFrame(rows).set_index("mode")

,n_train,adj_r2_in_sample,holdout_n,spearman_pred_vs_daytime_rate
mode,,,,
rate_within_city,2950,0.053,2064,0.350
rate,2950,0.022,2064,0.317
rate_daytime_within_city,2950,0.172,2064,0.526
rate_daytime,2950,0.224,2064,0.543
logcount,2950,0.375,2064,0.438


In [6]:
# peek at the headline (within-city daytime) fold's top predicted-risk BGs in the held-out city
scored_by_mode["rate_daytime_within_city"][["geoid", "city", "daytime_pop", "cl_total_count",
                                             "cl_total_rate_daytime", "y_pred"]] \
    .sort_values("y_pred", ascending=False).head(10)

,geoid,city,daytime_pop,cl_total_count,cl_total_rate_daytime,y_pred
3689,170318391001,chicago,275951.0,1581.0,5.729278,2.437890
2704,170313201021,chicago,42332.0,513.0,12.118492,1.235233
2615,170312712001,chicago,1386.0,98.0,70.707071,1.090210
2706,170313204001,chicago,44924.0,753.0,16.761642,1.037122
3690,170318391002,chicago,60357.0,280.0,4.639064,0.983016
3302,170316806001,chicago,1161.0,73.0,62.876830,0.979581
2907,170314608001,chicago,639.0,16.0,25.039124,0.965656
2599,170312601001,chicago,1664.0,69.0,41.466346,0.961725
3667,170318378001,chicago,3335.0,186.0,55.772114,0.945905
2803,170314207005,chicago,649.0,158.0,243.451464,0.933645


**Component 3: LOCO driver (all 5 folds)**

Rotating leave-one-city-out for each target mode: fit on the four train cities, score the
held-out city, repeat. `run_all_modes` returns `{mode: result}` where each result carries
`scored` (every BG's **out-of-sample** prediction) and per-fold `fits`.

In [7]:
from regression_modelling.models.cv import run_loco, run_all_modes, loco_metrics, plot_lorenz

# winsor_upper=250 gently caps whatever the daytime floor leaves in the top tail (a fixed
# constant, so it is leakage-free). run_all_modes now spans BOTH denominators:
#   rate_within_city, rate                 (plain population rate)
#   rate_daytime_within_city, rate_daytime (population + LODES jobs)  + logcount comparator
runs = run_all_modes(pooled, winsor_upper=250)


LOCO — target mode = 'rate_within_city' (cl_total), winsor@250
  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.320


  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.302
  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.342
  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.325


  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.302
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'rate' (cl_total), winsor@250
  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.342
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.313


  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.345
  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.356
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.321
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'rate_daytime_within_city' (cl_total), winsor@250


  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.250
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.177
  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.281


  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.301
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.243
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'rate_daytime' (cl_total), winsor@250
  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.287


  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.233
  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.268


  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.364
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.276
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'logcount' (cl_total), winsor@250


  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.401
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.375
  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.412


  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.428
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.390
  -> 5014 BGs scored out-of-sample across 5 folds


**Component 4: held-out metrics** --> Under Construction 

**Headline = concentration/Lorenz.** `gini` = model concentration (0 = no skill), `gini_oracle`
= best achievable, `skill` = gini/gini_oracle, `capture@20` = share of crime in the highest-risk
BGs covering 20% of population. `spearman` = rank corr of predicted score vs actual rate.
R²/RMSE/MAE appear only for the absolute `rate` mode.

In [8]:
# headline target: within-city standardized DAYTIME rate, daytime_pop-weighted x-axis
loco_metrics(runs["rate_daytime_within_city"], x_unit="daytime_pop")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.210,0.459,0.458,0.307,0.601
chicago,2064,0.076,0.436,0.175,0.176,0.527
detroit,600,0.079,0.324,0.243,0.226,0.407
houston,1536,0.123,0.484,0.253,0.220,0.443
kansas_city,445,0.191,0.445,0.429,0.255,0.592
POOLED,5014,0.084,0.454,0.186,0.182,0.464


In [9]:
# reported target (a): absolute DAYTIME rate — interpretable units, adds OOS R2/RMSE/MAE
loco_metrics(runs["rate_daytime"], x_unit="daytime_pop")

,n,gini,gini_oracle,skill,capture@20,spearman,r2_oos,rmse,mae
holdout,,,,,,,,,
atlanta,369,0.211,0.459,0.460,0.307,0.609,-3.486,48.63,41.93
chicago,2064,0.089,0.436,0.203,0.186,0.547,0.207,21.82,17.12
detroit,600,0.074,0.324,0.230,0.226,0.402,-0.039,23.14,17.89
houston,1536,0.128,0.484,0.264,0.228,0.464,0.099,29.44,18.56
kansas_city,445,0.211,0.445,0.474,0.285,0.639,0.287,22.35,15.20
POOLED,5014,0.099,0.454,0.218,0.205,0.497,-0.019,27.27,19.31


In [10]:
# comparator: log(count+1)
loco_metrics(runs["logcount"], x_unit="population")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.283,0.463,0.611,0.357,0.652
chicago,2064,0.276,0.445,0.620,0.374,0.532
detroit,600,0.161,0.292,0.550,0.291,0.445
houston,1536,0.329,0.520,0.633,0.388,0.600
kansas_city,445,0.387,0.517,0.748,0.447,0.708
POOLED,5014,0.284,0.470,0.605,0.367,0.561


In [11]:
# concentration curves per held-out city (headline daytime target)
plot_lorenz(runs["rate_daytime_within_city"], x_unit="daytime_pop")

<Axes: title={'center': 'LOCO concentration — rate_daytime_within_city (x=daytime_pop)'}, xlabel='cumulative share of block groups', ylabel='cumulative share of cl_total crime captured'>

#### x-axis sensitivity — population vs block-group

Population-weighting is the default (per-capita rate index). The BG-axis is the per-place
framing; compare to see how much the ranking verdict depends on the weighting choice.

In [12]:
loco_metrics(runs["rate_daytime_within_city"], x_unit="bg")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.188,0.458,0.411,0.253,0.601
chicago,2064,0.284,0.464,0.613,0.375,0.527
detroit,600,0.034,0.321,0.106,0.177,0.407
houston,1536,0.246,0.521,0.472,0.327,0.443
kansas_city,445,0.284,0.461,0.616,0.360,0.592
POOLED,5014,0.180,0.477,0.378,0.280,0.464


### Target stabilization — daytime denominator + floor + winsorize

The plain population rate is dominated by a few small-denominator artifacts (skew ≈ 32; see
`02_regression_inference`). The **daytime** denominator (population + LODES jobs) plus the
`daytime_pop ≥ 100` floor and a gentle winsorize give an interpretable rate target with a
sane scale. Score the daytime modes with the coherent `x_unit="daytime_pop"` weighting.

In [13]:
# absolute daytime rate (interpretable target) — adds out-of-sample R2/RMSE/MAE
loco_metrics(runs["rate_daytime"], x_unit="daytime_pop")

,n,gini,gini_oracle,skill,capture@20,spearman,r2_oos,rmse,mae
holdout,,,,,,,,,
atlanta,369,0.211,0.459,0.460,0.307,0.609,-3.486,48.63,41.93
chicago,2064,0.089,0.436,0.203,0.186,0.547,0.207,21.82,17.12
detroit,600,0.074,0.324,0.230,0.226,0.402,-0.039,23.14,17.89
houston,1536,0.128,0.484,0.264,0.228,0.464,0.099,29.44,18.56
kansas_city,445,0.211,0.445,0.474,0.285,0.639,0.287,22.35,15.20
POOLED,5014,0.099,0.454,0.218,0.205,0.497,-0.019,27.27,19.31


In [14]:
# within-city standardized daytime rate (relative BG risk, city level removed)
loco_metrics(runs["rate_daytime_within_city"], x_unit="daytime_pop")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.210,0.459,0.458,0.307,0.601
chicago,2064,0.076,0.436,0.175,0.176,0.527
detroit,600,0.079,0.324,0.243,0.226,0.407
houston,1536,0.123,0.484,0.253,0.220,0.443
kansas_city,445,0.191,0.445,0.429,0.255,0.592
POOLED,5014,0.084,0.454,0.186,0.182,0.464


In [15]:
# concentration curves per held-out city (daytime rate)
plot_lorenz(runs["rate_daytime"], x_unit="daytime_pop")

<Axes: title={'center': 'LOCO concentration — rate_daytime (x=daytime_pop)'}, xlabel='cumulative share of block groups', ylabel='cumulative share of cl_total crime captured'>

### Model-form probe — LightGBM (linearity vs feature ceiling)

Fit a gradient-boosted tree on the **same features, same LOCO folds, same stabilized target**
as the OLS harness — only the estimator changes. Reading:

- **GBM skill ≫ OLS skill** → *linearity is the bottleneck*: the features carry signal OLS
  can't reach (nonlinearity / interactions). Add splines/interactions or go nonlinear.
- **GBM skill ≈ OLS skill** → *features are the ceiling*: the linear form is fine; get better
  features.

Rows are matched to OLS (`dropna` on predictors) so the comparison is controlled. LightGBM
also gives gain-importance now and SHAP later. NB: `n_jobs=1` — `-1` oversubscribes threads
and hangs on this VM.

In [16]:
import numpy as np
import lightgbm as lgb
from regression_modelling.models.cv import make_target
from regression_modelling.constants import PREDICTOR_COLS

LGB_PARAMS = dict(
    n_estimators=400, learning_rate=0.03, num_leaves=31,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    min_child_samples=30, reg_lambda=1.0, random_state=0,
    n_jobs=1, verbosity=-1,   # n_jobs=1: -1 hangs on this VM (thread oversubscription)
)

def run_loco_gbm(pooled, mode="rate_daytime", winsor_upper=250, params=None,
                 predictors=PREDICTOR_COLS, category="cl_total"):
    """LOCO with LightGBM — same folds/features/target as OLS, so any skill gap is model
    FORM. Returns a run dict consumable by loco_metrics / plot_lorenz."""
    params = params or LGB_PARAMS
    parts, fits = {}, {}
    scored_parts = []
    print(f"LOCO-GBM — mode={mode!r} winsor@{winsor_upper}")
    for city, train, holdout in loco_folds(pooled):
        tr = train.copy()
        tr["_y"] = make_target(tr, mode, category, winsor_upper=winsor_upper)
        tr = tr.dropna(subset=list(predictors) + ["_y"])
        ho = holdout.dropna(subset=list(predictors)).copy()
        model = lgb.LGBMRegressor(**params).fit(tr[predictors], tr["_y"])
        ho["y_pred"] = model.predict(ho[predictors])
        fits[city] = model
        scored_parts.append(ho.assign(holdout_city=city))
        print(f"  holdout={city:14} n_train={len(tr):>5} scored={len(ho):>5}")
    scored = pd.concat(scored_parts, ignore_index=True)
    return {"scored": scored, "fits": fits, "mode": mode,
            "category": category, "predictors": list(predictors)}

gbm_run = run_loco_gbm(pooled, mode="rate_daytime", winsor_upper=250)
loco_metrics(gbm_run, x_unit="daytime_pop")

LOCO-GBM — mode='rate_daytime' winsor@250


  holdout=atlanta        n_train= 4645 scored=  369


  holdout=chicago        n_train= 2950 scored= 2064


  holdout=detroit        n_train= 4414 scored=  600


  holdout=houston        n_train= 3478 scored= 1536


  holdout=kansas_city    n_train= 4569 scored=  445


,n,gini,gini_oracle,skill,capture@20,spearman,r2_oos,rmse,mae
holdout,,,,,,,,,
atlanta,369,0.143,0.459,0.312,0.240,0.511,-0.034,23.34,18.43
chicago,2064,0.188,0.436,0.431,0.317,0.601,0.284,20.73,15.05
detroit,600,0.142,0.324,0.439,0.270,0.414,0.135,21.12,16.26
houston,1536,0.163,0.484,0.337,0.265,0.396,0.082,29.73,18.58
kansas_city,445,0.228,0.445,0.513,0.283,0.618,0.218,23.40,15.85
POOLED,5014,0.171,0.454,0.376,0.293,0.539,0.192,24.28,16.60


In [17]:
# OLS vs LightGBM on the SAME daytime target/folds — pooled headline comparison
def _pooled(run, label):
    row = loco_metrics(run, x_unit="daytime_pop").loc["POOLED"]
    return {"model": label, **row[["skill", "gini", "capture@20", "spearman", "r2_oos"]].to_dict()}

pd.DataFrame([
    _pooled(runs["rate_daytime"], "OLS (linear)"),
    _pooled(gbm_run,              "LightGBM"),
]).set_index("model")

,skill,gini,capture@20,spearman,r2_oos
model,,,,,
OLS (linear),0.218,0.099,0.205,0.497,-0.019
LightGBM,0.376,0.171,0.293,0.539,0.192


#### Standard metrics for the LightGBM model

Familiar yardsticks alongside the concentration metrics:

- **Regression accuracy** — out-of-sample `r2_oos`, `rmse`, `mae` on the daytime rate (same
  units as the target).
- **Ranking precision/recall** — label the true hotspots as the top-20% of BGs by *actual*
  crime count; the model flags the top-20% by predicted score. Both lists are the same size,
  so `precision == recall` (the hit rate). `capture@20%bg` = share of all crime those flagged
  BGs contain.

In [18]:
from regression_modelling.models.cv import error_stats, hotspot_metrics

# standard regression + ranking metrics for the LightGBM predictions, per held-out city + pooled
def standard_metrics(run, rate_col="cl_total_rate_daytime", category="cl_total", k=0.20):
    rows = []
    groups = list(run["scored"].groupby("holdout_city")) + [("POOLED", run["scored"])]
    for name, g in groups:
        d = {"holdout": name, "n": len(g)}
        d.update(error_stats(g, rate_col=rate_col))
        d.update(hotspot_metrics(g, category=category, k=k))
        rows.append(d)
    return pd.DataFrame(rows).set_index("holdout")

standard_metrics(gbm_run)

,n,r2_oos,rmse,mae,precision@20%bg,recall@20%bg,capture@20%bg
holdout,,,,,,,
atlanta,369,-0.034,23.34,18.43,0.378,0.378,0.312
chicago,2064,0.284,20.73,15.05,0.436,0.436,0.340
detroit,600,0.135,21.12,16.26,0.267,0.267,0.222
houston,1536,0.082,29.73,18.58,0.287,0.287,0.261
kansas_city,445,0.218,23.40,15.85,0.461,0.461,0.345
POOLED,5014,0.192,24.28,16.60,0.320,0.320,0.270


In [19]:
# LightGBM gain importance, averaged across LOCO folds (precursor to SHAP)
import matplotlib.pyplot as plt
imp = (pd.DataFrame({c: m.booster_.feature_importance("gain") for c, m in gbm_run["fits"].items()},
                    index=PREDICTOR_COLS)
       .mean(axis=1).sort_values())
ax = imp.plot.barh(figsize=(8, 8), color="#2c7fb8")
ax.set_title("LightGBM mean gain importance (avg over LOCO folds)")
ax.set_xlabel("gain"); plt.tight_layout(); plt.show()

### Actual vs predicted — is the model underpredicting?

Only meaningful for the ABSOLUTE rate modes (predictions in rate units). The 45° line is
perfect calibration; points **below** it are underpredicted. A **fit slope < 1** is the
signature of *shrinkage toward the mean* — the model compresses its predictions, so the
highest-crime BGs get severely underpredicted (it can't reach a held-out city's extremes
from static features).

In [20]:
import matplotlib.pyplot as plt

RC = "cl_total_rate_daytime"
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, (label, run) in zip(axes, [("OLS", runs["rate_daytime"]), ("LightGBM", gbm_run)]):
    s = run["scored"]; a = s[RC].to_numpy(float); p = s["y_pred"].to_numpy(float)
    ax.scatter(a, p, s=8, alpha=0.25, color="#2c7fb8")
    lim = max(a.max(), p.max()) * 1.02
    ax.plot([0, lim], [0, lim], "--", color="grey", lw=1, label="perfect (45°)")
    b, intc = np.polyfit(a, p, 1)                      # fitted pred-vs-actual line
    xs = np.array([0, lim]); ax.plot(xs, intc + b * xs, color="#c0392b", lw=1.6,
                                     label=f"fit slope={b:.2f}")
    under = (a > p).mean()
    top = s.nlargest(max(1, int(0.1 * len(s))), RC)
    ax.set_title(f"{label} — daytime rate\n{under:.0%} underpredicted | top-10%: "
                 f"act {top[RC].mean():.0f} vs pred {top['y_pred'].mean():.0f}")
    ax.set_xlabel("actual daytime rate (crimes/1k)"); ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("predicted"); plt.tight_layout(); plt.show()

### Denominator comparison — plain rate vs daytime rate (OLS & LightGBM)

Same features/folds; only the target denominator (and its coherent x-weighting) changes.
Watch two different questions pull in **different directions**: rank quality (`skill`,
`spearman`) vs absolute accuracy (`r2_oos`).

In [21]:
def _cmp_row(run, xu, model, den):
    r = loco_metrics(run, x_unit=xu).loc["POOLED"]
    return {"denominator": den, "model": model, "skill": r["skill"],
            "spearman": r["spearman"], "capture@20": r["capture@20"],
            "r2_oos": r.get("r2_oos", np.nan)}

ols_plain = run_loco(pooled, mode="rate", winsor_upper=250)
gbm_plain = run_loco_gbm(pooled, mode="rate", winsor_upper=250)

pd.DataFrame([
    _cmp_row(ols_plain,            "population",   "OLS",      "plain rate"),
    _cmp_row(gbm_plain,            "population",   "LightGBM", "plain rate"),
    _cmp_row(runs["rate_daytime"], "daytime_pop",  "OLS",      "daytime rate"),
    _cmp_row(gbm_run,              "daytime_pop",  "LightGBM", "daytime rate"),
]).set_index(["denominator", "model"])

LOCO — target mode = 'rate' (cl_total), winsor@250
  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.342
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.313
  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.345


  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.356
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.321
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO-GBM — mode='rate' winsor@250


  holdout=atlanta        n_train= 4645 scored=  369


  holdout=chicago        n_train= 2950 scored= 2064


  holdout=detroit        n_train= 4414 scored=  600


  holdout=houston        n_train= 3478 scored= 1536


  holdout=kansas_city    n_train= 4569 scored=  445


skill  spearman  capture@20  r2_oos
denominator  model                                        
plain rate   OLS       0.614     0.586       0.368   0.001
             LightGBM  0.583     0.568       0.360   0.002
daytime rate OLS       0.218     0.497       0.205  -0.019
             LightGBM  0.376     0.539       0.293   0.192